In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# 상담음성 (AI Hub, 이미 알고 있는 경로)
aihub_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성"
print("상담음성 존재?", os.path.exists(aihub_path))

# his_voice - 바로가기 추가했으니 MyDrive 아래에서 폴더명으로 찾아야 함
print("\nMyDrive 최상위 폴더 목록:")
print(os.listdir("/content/drive/MyDrive"))

Mounted at /content/drive
상담음성 존재? True

MyDrive 최상위 폴더 목록:
['사진들', '공부 프롬프트.txt', 'ai 휴먼', '주식', '크롬 저장', '3. Finetuning', '07_자율주행_도메인적응과_PEFT_LoRA_직관.ipynb의 사본', '08_RLHF_첫걸음_Reward_Model과_PPO.ipynb의 사본', '06_Unsloth_실전_LLaMA와_RAG비교.ipynb의 사본', '협업의_기술_The_Art_of_Collaboration_insight_v3_expanded .pdf의 사본.pdf', 'Colab Notebooks', 'voice_pishing_test', 'aihub data', 'voice_pishing_text_merged', '저음질_D03_결과.zip', 'his_voice']


In [ ]:
his_voice_path = "/content/drive/MyDrive/his_voice"
print("his_voice 존재?", os.path.exists(his_voice_path))
print("\nhis_voice 내부:")
print(os.listdir(his_voice_path))

his_voice 존재? True

his_voice 내부:
['fss_impersonation_run_20260813_161023.err.log', 'download_fss_impersonation.py', 'download_fss_audio.py', 'fss_impersonation_run_20260813_161023.log', 'data', 'docs', 'tests', '__pycache__', 'stt_finetuning_cleaned_v2.zip', 'stt_finetuning_cleaned_v3.zip']


In [ ]:
data_path = os.path.join(his_voice_path, "data")
print("data 폴더 내부:")
print(os.listdir(data_path))

data 폴더 내부:
['impersonation', 'loanScam']


In [ ]:
impersonation_path = os.path.join(data_path, "impersonation")
loan_scam_path = os.path.join(data_path, "loanScam")

print("impersonation 내부 (앞 10개):")
print(os.listdir(impersonation_path)[:10])
print("총 개수:", len(os.listdir(impersonation_path)))

print("\nloanScam 내부 (앞 10개):")
print(os.listdir(loan_scam_path)[:10])
print("총 개수:", len(os.listdir(loan_scam_path)))

impersonation 내부 (앞 10개):
['36703_다운로드받으신 보안프로그램이 있으신가요_그런건 국가에서 인정하지 않은 개인 보안프~ 정부 앱과 충돌할 수 있기 때문에 기존에 다운받으신 보안프로그램은 모두 다 삭제해주세요.mp3', '36703_다운로드받으신 보안프로그램이 있으신가요_그런건 국가에서 인정하지 않은 개인 보안프~ 정부 앱과 충돌할 수 있기 때문에 기존에 다운받으신 보안프로그램은 모두 다 삭제해주세요.txt', '36643_(2) 은행권에 들어갔다 나오더니 태도가 갑자기 왜 이렇습니까_본인 관할 경찰서 어디에요.txt', '36643_(2) 은행권에 들어갔다 나오더니 태도가 갑자기 왜 이렇습니까_본인 관할 경찰서 어디에요.mp3', '36745_금융범죄수사중입니다. 사건내용을 유출할 경우 처벌받을수있습니다.mp3', '36740_본인과 연루된 불법 명의도용사건이 검찰에 접수되어 연락드렸습니다.mp3', '36739_다음주에 재판이 열립니다. 재판증거자료를 위해 통화를 녹취하겠습니다.mp3', '36736_강남서 사이버수사과입니다.mp3',

In [ ]:
def count_by_extension(folder_path):
    files = os.listdir(folder_path)
    txt_files = [f for f in files if f.endswith('.txt')]
    mp3_files = [f for f in files if f.endswith('.mp3')]
    return txt_files, mp3_files

imp_txt, imp_mp3 = count_by_extension(impersonation_path)
loan_txt, loan_mp3 = count_by_extension(loan_scam_path)

print("impersonation - txt:", len(imp_txt), "/ mp3:", len(imp_mp3))
print("loanScam - txt:", len(loan_txt), "/ mp3:", len(loan_mp3))

# txt 없는 mp3 파일이 있는지 확인 (파일명에서 확장자만 다르고 나머지는 같아야 쌍)
imp_txt_basenames = set(f[:-4] for f in imp_txt)  # .txt 제거
imp_mp3_basenames = set(f[:-4] for f in imp_mp3)  # .mp3 제거

missing_txt = imp_mp3_basenames - imp_txt_basenames
print(f"\nimpersonation - txt 없는 mp3 개수: {len(missing_txt)}")
if missing_txt:
    print("예시:", list(missing_txt)[:5])

impersonation - txt: 94 / mp3: 237
loanScam - txt: 145 / mp3: 185

impersonation - txt 없는 mp3 개수: 143
예시: ['36491_피해자 입증 절차를 밟아야 고소 취하됩니다', '36456_저희 검찰청에서 국제사기단 김인수 일당을 검거했다', '36533_[검찰사칭] 잠시 자리 이동해서 통화 가능하세요_', '36495_최근 2,3년 사이에 핸드폰이나 지갑, 신분증을 분실하신 경험', '36466_[사기 시나리오 3단계] ATM계좌 이체한도 초과1']


In [ ]:
print("1. 사기 데이터 로딩 시작...")
fraud_data = load_txt_files(impersonation_path, label=1)
fraud_data += load_txt_files(loan_scam_path, label=1)
print("사기 데이터 개수:", len(fraud_data))

print("2. AI Hub 파일 목록 불러오는 중... (시간 걸릴 수 있음)")
aihub_files = [f for f in os.listdir(aihub_path) if f.endswith('.txt')]
print("AI Hub 파일 목록 완료, 총 개수:", len(aihub_files))

print("3. 정상 데이터 샘플링 중...")
random.seed(42)
sampled_files = random.sample(aihub_files, min(400, len(aihub_files)))

normal_data = []
for fname in sampled_files:
    path = os.path.join(aihub_path, fname)
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    normal_data.append({'text': text, 'label': 0, 'source': fname})

print("정상 데이터 개수:", len(normal_data))

1. 사기 데이터 로딩 시작...
사기 데이터 개수: 239
2. AI Hub 파일 목록 불러오는 중... (시간 걸릴 수 있음)
AI Hub 파일 목록 완료, 총 개수: 18520
3. 정상 데이터 샘플링 중...
정상 데이터 개수: 400


In [ ]:
import zipfile
import time

zip_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성_결과.zip"
extract_path = "/content/상담음성_전체"

start = time.time()
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("압축 해제 완료, 걸린 시간:", time.time() - start, "초")

print(os.listdir(extract_path))

압축 해제 완료, 걸린 시간: 5.9299163818359375 초
['S00000333_merged.txt', 'S00009361_merged.txt', 'S00015313_merged.txt', 'S00002802_merged.txt', 'S00003934_merged.txt', 'S00009670_merged.txt', 'S00005971_merged.txt', 'S00012383_merged.txt', 'S00004565_merged.txt', 'S00004158_merged.txt', 'S00010868_merged.txt', 'S00004798_merged.txt', 'S00011169_merged.txt', 'S00010836_merged.txt', 'S00013750_merged.txt', 'S00011912_merged.txt', 'S00002833_merged.txt', 'S00003320_merged.txt', 'S00014703_merged.txt', 'S00008371_merged.txt', 'S00008271_merged.txt', 'S00008875_merged.txt', 'S00005130_merged.txt', 'S00016231_merged.txt', 'S00013217_merged.txt', 'S00009556_merged.txt', 'S00008579_merged.txt', 'S00009241_merged.txt', 'S00004432_merged.txt', 'S00010915_merged.txt', 'S00001415_merged.txt', 'S00007918_merged.txt', 'S00016566_merged.txt', 'S00005891_merged.txt', 'S00001132_merged.txt', 'S00002888_merged.txt', 'S00002664_merged.txt', 'S00016842_merged.txt', 'S00010892_merged.txt', 'S00000970_merged.txt', '

In [ ]:
files = os.listdir(extract_path)
txt_files = [f for f in files if f.endswith('.txt')]
print("전체 파일 수:", len(files))
print("txt 파일 수:", len(txt_files))

전체 파일 수: 18521
txt 파일 수: 18520


In [ ]:
# 1. _merge_summary.json 내용 확인 - 병합 당시 기록된 실제 개수
import json

summary_path = os.path.join(extract_path, "_merge_summary.json")
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)
print(summary)

{'dataset': '상담음성', 'total_sessions': 49839, 'success_count': 49839, 'failed_count': 0, 'total_time_minutes': 7.787224364280701, 'output_dir': '/content/drive/MyDrive/voice_pishing_text_merged/상담음성'}


In [ ]:
# 2. txt 파일 하나 내용 열어서 실제로 상담음성 내용인지 확인
sample_file = os.path.join(extract_path, txt_files[0])
with open(sample_file, "r", encoding="utf-8") as f:
    content = f.read()
print(content[:500])

안녕하세요. 셰어링 에듀 상담사 정서경입니다.
네 안녕하세요. 이번에 나온 영단어 암기하는 기기 이름이 뭐였죠?
뇌새김입니다.
아/ 그/ 뇌새김이요?
뭐/ 광고에서는 (1)/(일) 분에 (10)/(열)단어 정도 외울 수 있다고 하던데 사실인가요? 제가 기억력이 나쁜 편이어서 할 수 있을지 의심되네요.
그거 출시일이 언제죠? 제 광고에서 보기로는 이번 달 말인가 그랬던것 같은데. 가능하면 지금 바로 구매하려고 했거든요.
내일입니다.
그럼 물품 수량은 충분한 거죠? 조기에 다 팔려서 재고가 없거나+ 재고가 없거나해서 오래 기다려야 할 일은 없었으면 해서요.
그럴 수도 있습니다.
아/ 그래요? 그럼 몇 시부터 판매 개시하나요? 제가 제때 못 사면 못참는 성격이라 바로 사고 싶어서요. 만약 출근 시간이랑 겹치는 건 아니겠죠?
아침 (10)/(열)시요.
아/ 그때 출근해야 되는데 혹시 가족이나 지인이 대신 가서 구매할 수 있는거죠?
그럼 그렇게 할게요. 아/ 그리고 궁금한게 있었는데 그거 색


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# 두 데이터 합치기
all_data = fraud_data + normal_data
df = pd.DataFrame(all_data)

print("전체 개수:", len(df))
print(df['label'].value_counts())

# Train/Validation 나누기 (80:20, 라벨 비율 유지)
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

print("\nTrain 개수:", len(train_df))
print("Validation 개수:", len(val_df))

전체 개수: 639
label
0    400
1    239
Name: count, dtype: int64

Train 개수: 511
Validation 개수: 128


In [ ]:
from transformers import ElectraTokenizer, ElectraForSequenceClassification

tokenizer = ElectraTokenizer.from_pretrained("skplanet/dialog-koelectra-small-discriminator")
model = ElectraForSequenceClassification.from_pretrained(
    "skplanet/dialog-koelectra-small-discriminator",
    num_labels=2  # 정상(0) / 사기(1) 이진 분류
)

tokenizer_config.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/319k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 58.9MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: skplanet/dialog-koelectra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on yo

In [15]:
def tokenize_function(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_df['text'].tolist())
val_encodings = tokenize_function(val_df['text'].tolist())

print(train_encodings['input_ids'].shape)
print(val_encodings['input_ids'].shape)

torch.Size([511, 256])
torch.Size([128, 256])


In [16]:
import torch

class VoicePhishingDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = VoicePhishingDataset(train_encodings, train_df['label'].tolist())
val_dataset = VoicePhishingDataset(val_encodings, val_df['label'].tolist())

print("Train dataset 크기:", len(train_dataset))
print("Validation dataset 크기:", len(val_dataset))

Train dataset 크기: 511
Validation dataset 크기: 128


In [17]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'logging_dir'

In [18]:
!pip install transformers==4.44.2 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. 데이터 로딩 함수 및 실행 (아까 만든 것 그대로)
import os

def load_txt_files(folder_path, label):
    data = []
    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    for fname in files:
        path = os.path.join(folder_path, fname)
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        data.append({'text': text, 'label': label, 'source': fname})
    return data

impersonation_path = "/content/drive/MyDrive/his_voice/data/impersonation"
loan_scam_path = "/content/drive/MyDrive/his_voice/data/loanScam"
aihub_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성"

fraud_data = load_txt_files(impersonation_path, label=1)
fraud_data += load_txt_files(loan_scam_path, label=1)

import random
aihub_files = [f for f in os.listdir(aihub_path) if f.endswith('.txt')]
random.seed(42)
sampled_files = random.sample(aihub_files, min(400, len(aihub_files)))

normal_data = []
for fname in sampled_files:
    path = os.path.join(aihub_path, fname)
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    normal_data.append({'text': text, 'label': 0, 'source': fname})

# 3. Train/Val 나누기
import pandas as pd
from sklearn.model_selection import train_test_split

all_data = fraud_data + normal_data
df = pd.DataFrame(all_data)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# 4. 모델 & 토크나이저
from transformers import ElectraTokenizer, ElectraForSequenceClassification

tokenizer = ElectraTokenizer.from_pretrained("skplanet/dialog-koelectra-small-discriminator")
model = ElectraForSequenceClassification.from_pretrained(
    "skplanet/dialog-koelectra-small-discriminator",
    num_labels=2
)

# 5. 토큰화
def tokenize_function(texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=256, return_tensors="pt")

train_encodings = tokenize_function(train_df['text'].tolist())
val_encodings = tokenize_function(val_df['text'].tolist())

# 6. Dataset 클래스
import torch

class VoicePhishingDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = VoicePhishingDataset(train_encodings, train_df['label'].tolist())
val_dataset = VoicePhishingDataset(val_encodings, val_df['label'].tolist())

print("여기까지 완료. Train:", len(train_dataset), "Val:", len(val_dataset))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of ElectraForSequenceClassification were not initialized from the model checkpo

여기까지 완료. Train: 511 Val: 128


In [2]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer 준비 완료")

Trainer 준비 완료


In [3]:
import zipfile

zip_path = "/content/drive/MyDrive/저음질_D03_결과.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    names = zip_ref.namelist()
    txt_in_zip = [n for n in names if n.endswith('.txt')]

print("zip 안에 있는 txt 파일 개수:", len(txt_in_zip))

zip 안에 있는 txt 파일 개수: 8335


In [4]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


RuntimeError: The size of tensor a (256) must match the size of tensor b (128) at non-singleton dimension 1

In [5]:
def tokenize_function(texts):
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")

train_encodings = tokenize_function(train_df['text'].tolist())
val_encodings = tokenize_function(val_df['text'].tolist())

train_dataset = VoicePhishingDataset(train_encodings, train_df['label'].tolist())
val_dataset = VoicePhishingDataset(val_encodings, val_df['label'].tolist())

print("재토큰화 완료. Train:", len(train_dataset), "Val:", len(val_dataset))

재토큰화 완료. Train: 511 Val: 128


In [6]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

import os
os.environ["WANDB_DISABLED"] = "true"  # 아까 그 팝업 다시 안 뜨게

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.464600,0.358185,1.000000,1.000000,1.000000,1.000000


ValueError: You are trying to save a non contiguous tensor: `electra.embeddings_project.weight` which is not allowed. It either means you are trying to save tensors which are reference of each other in which case it's recommended to save only the full tensors, and reslice at load time, or simply call `.contiguous()` on your tensor to pack it before saving.

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    save_safetensors=False,   # 추가된 부분
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

import os
os.environ["WANDB_DISABLED"] = "true"

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.165100,0.113324,1.000000,1.000000,1.000000,1.000000
2,0.074700,0.055125,1.000000,1.000000,1.000000,1.000000
3,0.053200,0.045121,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=96, training_loss=0.12631067043791214, metrics={'train_runtime': 14.0902, 'train_samples_per_second': 108.799, 'train_steps_per_second': 6.813, 'total_flos': 11275077348864.0, 'train_loss': 0.12631067043791214, 'epoch': 3.0})

In [8]:
import torch.nn.functional as F

model.eval()
predictions_info = []

with torch.no_grad():
    for i in range(len(val_dataset)):
        item = val_dataset[i]
        input_ids = item['input_ids'].unsqueeze(0).to(model.device)
        attention_mask = item['attention_mask'].unsqueeze(0).to(model.device)
        label = item['labels'].item()

        output = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(output.logits, dim=1)
        confidence = probs.max().item()
        predicted = probs.argmax().item()

        predictions_info.append({
            'true_label': label,
            'predicted': predicted,
            'confidence': confidence,
            'text': val_df.iloc[i]['text'][:100]
        })

import pandas as pd
result_df = pd.DataFrame(predictions_info)
print("평균 확신도:", result_df['confidence'].mean())
print("확신도 최소값:", result_df['confidence'].min())
print(result_df['confidence'].describe())

평균 확신도: 0.9558854885399342
확신도 최소값: 0.9449467658996582
count    128.000000
mean       0.955885
std        0.002788
min        0.944947
25%        0.953837
50%        0.957704
75%        0.958128
max        0.958521
Name: confidence, dtype: float64


In [9]:
# Train과 Validation에 겹치는 데이터(source 파일명)가 있는지 확인
train_sources = set(train_df['source'])
val_sources = set(val_df['source'])

overlap = train_sources & val_sources
print("Train/Validation 겹치는 파일 개수:", len(overlap))

Train/Validation 겹치는 파일 개수: 0


In [10]:
# 텍스트가 실제로 뭔지 다시 확인 (음성이 아니라 텍스트인지)
print(type(train_df['text'].iloc[0]))
print(train_df['text'].iloc[0][:200])

<class 'str'>
n/ 네. 안녕하세요,
n/ 제가 (1주일)/(일주일) 전에 백화점 지하 영플라자에서 폭스 퍼 오리털 패딩 (검은색)/(검정색)을 구매했었는데요. 이 상품에
n/ 영수증은 제가 어제 가방 뒤져+ 뒤져서 찾아놓긴 했는데, 상품을 구매하고 집에 오자마자 (태그)/(텍)을 떼 버려서요.
n/ 이걸 반품하고 싶기는 한데 가능할까요? 옷은 (태그)/(텍)만 떼고, 


In [11]:
import re

def merge_consecutive_dual_notation(text: str) -> str:
    pattern = r'((?:\(\d+\)/\([^)]*\)\s*에?\s*){2,}\(\d+\)/\([^)]*\))'
    def merge_group(match):
        numbers = re.findall(r'\((\d+)\)/\([^)]*\)', match.group(1))
        return ''.join(numbers)
    return re.sub(pattern, merge_group, text)

def normalize_dual_notation(text: str) -> str:
    pattern = r'\((\d[\w]*)\)/\([^)]*\)'
    return re.sub(pattern, r'\1', text)

def merge_spaced_digits(text: str) -> str:
    pattern = r'(?:\d\s){4,}\d'
    return re.sub(pattern, lambda m: m.group(0).replace(' ', ''), text)

def remove_correction_marker(text: str) -> str:
    return text.replace('+', ' ')

def remove_noise_markup(text: str) -> str:
    pattern = r'\(\([^)]*\)\)'
    return re.sub(pattern, ' ', text)

def remove_aihub_speaker_markup(text: str) -> str:
    pattern = r'\b[a-z]/'
    return re.sub(pattern, ' ', text)

def remove_masking_marker(text: str) -> str:
    pattern = r'\(삐-?\)'
    return re.sub(pattern, ' ', text)

def normalize_speaker_label(text: str) -> str:
    text = re.sub(r'(피해자|사기범)\s*:\s*', r'\1: ', text)
    return text

def clean_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_finance_text(text: str) -> str:
    text = merge_consecutive_dual_notation(text)
    text = normalize_dual_notation(text)
    text = merge_spaced_digits(text)
    text = remove_correction_marker(text)
    text = remove_noise_markup(text)
    text = remove_aihub_speaker_markup(text)
    text = remove_masking_marker(text)
    text = normalize_speaker_label(text)
    text = clean_whitespace(text)
    return text

print("정규화 함수 로딩 완료")

정규화 함수 로딩 완료


In [12]:
# df는 기존에 만든 raw 데이터 (fraud_data + normal_data)
df['text_normalized'] = df['text'].apply(normalize_finance_text)

# 정규화 전후 비교 샘플 확인
print("=== 정규화 전후 비교 (AI Hub 샘플) ===")
print("원본:", df[df['label']==0]['text'].iloc[0][:150])
print("정규화:", df[df['label']==0]['text_normalized'].iloc[0][:150])
print()
print("=== 정규화 전후 비교 (his_voice 샘플) ===")
print("원본:", df[df['label']==1]['text'].iloc[0][:150])
print("정규화:", df[df['label']==1]['text_normalized'].iloc[0][:150])

# 같은 시드로 다시 Train/Val 분할 (아까와 동일한 샘플 구성 유지)
train_df2, val_df2 = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print("\nTrain:", len(train_df2), "Val:", len(val_df2))

=== 정규화 전후 비교 (AI Hub 샘플) ===
원본: n/ 안녕하세요. 제주은행입니다.
n/ 네, 안녕하세요. 제주은행 고객센터 맞나요?
n/ 네. 맞습니다. 제주은행 고객센터 상담사 감귤입니다.
n/ 아/ 제가 이번에 재테크 고민이 좀 많아서 제주은행 상품들 상담 좀 받으려고 전화드렸어요.
n/ 그냥 평범한 직장인이라 
정규화: 안녕하세요. 제주은행입니다. 네, 안녕하세요. 제주은행 고객센터 맞나요? 네. 맞습니다. 제주은행 고객센터 상담사 감귤입니다. 아/ 제가 이번에 재테크 고민이 좀 많아서 제주은행 상품들 상담 좀 받으려고 전화드렸어요. 그냥 평범한 직장인이라 따로 재테크 고민을 해보지 

=== 정규화 전후 비교 (his_voice 샘플) ===
원본: 피해자 : 예 아 그러니까 김명식 이란 사람이 제 주민 번호랑
사기범 : 네 네 아 통장 개설해가지고 지금 대포 통장으로 사용됐다는 말씀입니다 지금.
피해자 : 저한테 백오십만원 통장 하나당 주고.
사기범 : 네네 통장 구매했다고 지금 현재 진술을 하고 있어요
피해자 
정규화: 피해자: 예 아 그러니까 김명식 이란 사람이 제 주민 번호랑 사기범: 네 네 아 통장 개설해가지고 지금 대포 통장으로 사용됐다는 말씀입니다 지금. 피해자: 저한테 백오십만원 통장 하나당 주고. 사기범: 네네 통장 구매했다고 지금 현재 진술을 하고 있어요 피해자: 언제 

Train: 511 Val: 128


In [13]:
import random

random.seed(42)
normal_data_239 = random.sample(normal_data, 239)

print("정상:", len(normal_data_239), "사기:", len(fraud_data))

# 5:5 데이터프레임 구성
all_data_balanced = fraud_data + normal_data_239
df_balanced = pd.DataFrame(all_data_balanced)

print(df_balanced['label'].value_counts())

# 정규화 적용
df_balanced['text_normalized'] = df_balanced['text'].apply(normalize_finance_text)

# Train/Val 분할
train_df3, val_df3 = train_test_split(
    df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label']
)

print("Train:", len(train_df3), "Val:", len(val_df3))

정상: 239 사기: 239
label
1    239
0    239
Name: count, dtype: int64
Train: 382 Val: 96


In [14]:
# 토큰화
train_encodings3 = tokenizer(
    train_df3['text_normalized'].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)
val_encodings3 = tokenizer(
    val_df3['text_normalized'].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)

train_dataset3 = VoicePhishingDataset(train_encodings3, train_df3['label'].tolist())
val_dataset3 = VoicePhishingDataset(val_encodings3, val_df3['label'].tolist())

# 모델 새로 초기화 (공정한 비교를 위해)
model3 = ElectraForSequenceClassification.from_pretrained(
    "skplanet/dialog-koelectra-small-discriminator",
    num_labels=2
)

trainer3 = Trainer(
    model=model3,
    args=training_args,
    train_dataset=train_dataset3,
    eval_dataset=val_dataset3,
    compute_metrics=compute_metrics,
)

trainer3.train()

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at skplanet/dialog-koelectra-small-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.588100,0.451647,1.000000,1.000000,1.000000,1.000000
2,0.375400,0.277073,1.000000,1.000000,1.000000,1.000000
3,0.251400,0.228283,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=72, training_loss=0.41499054183562595, metrics={'train_runtime': 9.3138, 'train_samples_per_second': 123.044, 'train_steps_per_second': 7.73, 'total_flos': 8428727098368.0, 'train_loss': 0.41499054183562595, 'epoch': 3.0})

In [15]:
# AI Hub 데이터에 '피해자' 또는 '사기범'이라는 단어가 있는지 확인
aihub_normalized = df_balanced[df_balanced['label']==0]['text_normalized']
his_voice_normalized = df_balanced[df_balanced['label']==1]['text_normalized']

aihub_has_label = aihub_normalized.str.contains('피해자|사기범').sum()
his_voice_has_label = his_voice_normalized.str.contains('피해자|사기범').sum()

print(f"AI Hub 중 '피해자'/'사기범' 포함: {aihub_has_label} / {len(aihub_normalized)}")
print(f"his_voice 중 '피해자'/'사기범' 포함: {his_voice_has_label} / {len(his_voice_normalized)}")

AI Hub 중 '피해자'/'사기범' 포함: 0 / 239
his_voice 중 '피해자'/'사기범' 포함: 196 / 239


In [16]:
def strip_speaker_label(text: str) -> str:
    """
    '피해자:', '사기범:' 라벨을 완전히 제거 (단어 자체를 삭제).

    이유: 라벨 존재 여부 자체가 데이터 누수를 일으킴이 확인됨
    (2026-08-19 실험 — AI Hub 0%, his_voice 82%가 이 단어를 포함,
     모델이 이것만으로 100% 분류했을 가능성 높음)
    """
    text = re.sub(r'(피해자|사기범)\s*:\s*', ' ', text)
    return text

# 정규화된 텍스트에서 라벨까지 마저 제거
df_balanced['text_no_label'] = df_balanced['text_normalized'].apply(strip_speaker_label)

# 확인
print(df_balanced[df_balanced['label']==1]['text_no_label'].iloc[0][:200])

# 다시 확인: 라벨이 진짜 다 지워졌는지
remaining = df_balanced['text_no_label'].str.contains('피해자|사기범').sum()
print("\n라벨 남은 개수:", remaining)

 예 아 그러니까 김명식 이란 사람이 제 주민 번호랑  네 네 아 통장 개설해가지고 지금 대포 통장으로 사용됐다는 말씀입니다 지금.  저한테 백오십만원 통장 하나당 주고.  네네 통장 구매했다고 지금 현재 진술을 하고 있어요  언제 구매 했대요 저한테?  작년이죠. 작년 구월 십팔일 날에  작년 구월 십팔일?  네 작년 구월 십팔일 날에 저희 쪽에서 이 김

라벨 남은 개수: 52


In [17]:
# 라벨이 아직 남아있는 샘플 찾아서 실제로 확인
remaining_mask = df_balanced['text_no_label'].str.contains('피해자|사기범')
remaining_samples = df_balanced[remaining_mask]['text_no_label']

print("남은 샘플 개수:", len(remaining_samples))
print("\n첫 번째 샘플 전체 확인:")
print(remaining_samples.iloc[0])

남은 샘플 개수: 52

첫 번째 샘플 전체 확인:
 예 아 그러니까 김명식 이란 사람이 제 주민 번호랑  네 네 아 통장 개설해가지고 지금 대포 통장으로 사용됐다는 말씀입니다 지금.  저한테 백오십만원 통장 하나당 주고.  네네 통장 구매했다고 지금 현재 진술을 하고 있어요  언제 구매 했대요 저한테?  작년이죠. 작년 구월 십팔일 날에  작년 구월 십팔일?  네 작년 구월 십팔일 날에 저희 쪽에서 이 김명식을 검거 했습니다.  아 작년 2016 구월 십팔일날 검거를 했는데 조사를 하던 중 이게 나왔다.  네 네 그니까 사람 많다 보니까 OOO씨 순서가 되니까 이렇게 연락드린 겁니다.  아 많아서. 예 알겠습니다.  예 그럼 일단 전화 끊지 마시구요. 제가 바로 내선으로 저희 담당 검사 님 앞으로 연결해 드릴게요.  네 네  예 잠시만요  네 여보세요  네  네 수원 지방 검찰청에 특수 일부 김기동 수사관입니다.  네  네 사건 번호가 어떻게 되시죠.  2017 조사 0668 김명식 명의 도용  잠시만 기다려주세요.  네  OOO씨 되시죠?  네  사건 내용에 대해서는 뭐 이해를 하신 거예요?  근데 제가 그 뭐 잃어 버리거나 그런 적이 없는데. 지금 기가 막힌다는 생각이 듭니다.  음 그니까 그 개인 정보에 대해서 잃어버린 것도 그런 것도 없으신데 어떻게 인해서 명의가 유출되느냐 이런 부분에 대해서 말씀하는 거죠.  주민 등록증을 잃어 버린게 20년 된거 같아요. 출근할 때 떨어 트린적이 한번있는데.  최근 일 년 안으로 뭐 개인 정보 유출로 인해서 뭐 피해 보신 적 있으십니까?  개인 정보 유출?  그 그 직접적인 피해가 가신 적 있나 여쭤보는 겁니다.  일 이년 사이엔 없는거 같아요.  일 이년 사이엔 없으시고. 그리고 그 최근 삼 년안으로 뭐 해외나 간 적 없으시고?  네 네  가까운 제주도 쪽에도 가신적 없으시고?  삼년 안으론 안 간 것 같은데요?  삼년 안으로 가신적은 없으시고.  혹시 그 기존에 그 농협 은행을 거래하시면서 뭐 혹시 이하나 직원

In [18]:
# 라벨 완전 제거된 데이터로 토큰화
train_df4, val_df4 = train_test_split(
    df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label']
)

train_encodings4 = tokenizer(
    train_df4['text_no_label'].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)
val_encodings4 = tokenizer(
    val_df4['text_no_label'].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)

train_dataset4 = VoicePhishingDataset(train_encodings4, train_df4['label'].tolist())
val_dataset4 = VoicePhishingDataset(val_encodings4, val_df4['label'].tolist())

# 모델 새로 초기화
model4 = ElectraForSequenceClassification.from_pretrained(
    "skplanet/dialog-koelectra-small-discriminator",
    num_labels=2
)

trainer4 = Trainer(
    model=model4,
    args=training_args,
    train_dataset=train_dataset4,
    eval_dataset=val_dataset4,
    compute_metrics=compute_metrics,
)

trainer4.train()

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at skplanet/dialog-koelectra-small-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.624200,0.497700,1.000000,1.000000,1.000000,1.000000
2,0.425300,0.327584,1.000000,1.000000,1.000000,1.000000
3,0.290500,0.269164,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=72, training_loss=0.45318397548463607, metrics={'train_runtime': 6.8438, 'train_samples_per_second': 167.451, 'train_steps_per_second': 10.521, 'total_flos': 8428727098368.0, 'train_loss': 0.45318397548463607, 'epoch': 3.0})

In [19]:
import os

# 원본 AI Hub 상담음성 압축 파일 경로 확인 (실제 경로로 바꿔줘)
original_path = "/content/drive/MyDrive/aihub data"  # 아까 목록에 있던 폴더
print(os.listdir(original_path))

['상담 음성', '저음질']


In [20]:
consulting_path = os.path.join(original_path, "상담 음성")
print(os.listdir(consulting_path))

['Training', 'Validation']


In [21]:
training_path = os.path.join(consulting_path, "Training")
print("Training 내부:", os.listdir(training_path))

validation_path = os.path.join(consulting_path, "Validation")
print("Validation 내부:", os.listdir(validation_path))

Training 내부: ['[라벨]KtelSpeech_train_D60_label_0.zip', '[라벨]KtelSpeech_train_D60_label_1.zip', '[라벨]KtelSpeech_train_D61_label_0.zip', '[라벨]KtelSpeech_train_D61_label_1.zip', '[라벨]KtelSpeech_train_D62_label_0.zip', '[라벨]KtelSpeech_train_D62_label_1.zip']
Validation 내부: ['[라벨]KtelSpeech_valid_D60_label_0.zip', '[라벨]KtelSpeech_valid_D61_label_0.zip', '[라벨]KtelSpeech_valid_D62_label_0.zip']


In [22]:
import zipfile

label0_zip = os.path.join(training_path, "[라벨]KtelSpeech_train_D60_label_0.zip")
label1_zip = os.path.join(training_path, "[라벨]KtelSpeech_train_D60_label_1.zip")

with zipfile.ZipFile(label0_zip, 'r') as zf:
    names0 = zf.namelist()
    print("label_0 개수:", len(names0))
    print("label_0 앞 5개:", names0[:5])
    print("label_0 뒤 5개:", names0[-5:])

with zipfile.ZipFile(label1_zip, 'r') as zf:
    names1 = zf.namelist()
    print("\nlabel_1 개수:", len(names1))
    print("label_1 앞 5개:", names1[:5])
    print("label_1 뒤 5개:", names1[-5:])

label_0 개수: 325111
label_0 앞 5개: ['D60/', 'D60/J91/', 'D60/J91/S00000001/', 'D60/J91/S00000002/', 'D60/J91/S00000003/']
label_0 뒤 5개: ['D60/J91/S00007877/0001.txt', 'D60/J91/S00007877/0002.txt', 'D60/J91/S00007877/0003.txt', 'D60/J91/S00007877/0045.txt', 'D60/J91/S00007877/0024.txt']

label_1 개수: 298920
label_1 앞 5개: ['D60/', 'D60/J91/', 'D60/J91/S00007878/', 'D60/J91/S00007879/', 'D60/J91/S00007880/']
label_1 뒤 5개: ['D60/J91/S00015775/0023.txt', 'D60/J91/S00015775/0001.txt', 'D60/J91/S00015775/0002.txt', 'D60/J91/S00015775/0003.txt', 'D60/J91/S00015775/0024.txt']


In [23]:
import zipfile

for d in ["D60", "D61", "D62"]:
    zip_path = os.path.join(training_path, f"[라벨]KtelSpeech_train_{d}_label_0.zip")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        names = zf.namelist()
        print(f"{d} label_0 개수: {len(names)}")
        print(f"{d} 앞 3개: {names[:3]}")
        print()

D60 label_0 개수: 325111
D60 앞 3개: ['D60/', 'D60/J91/', 'D60/J91/S00000001/']

D61 label_0 개수: 328399
D61 앞 3개: ['D61/', 'D61/J92/', 'D61/J92/S00000001/']

D62 label_0 개수: 242130
D62 앞 3개: ['D62/', 'D62/J93/', 'D62/J93/S00000001/']



In [24]:
import zipfile
import os
from collections import defaultdict
import re

def merge_category(training_path, category, subfolder):
    """D60/D61/D62 하나의 카테고리를 병합"""
    sessions = defaultdict(dict)  # {고유세션키: {파일번호: 텍스트}}

    for label in ["label_0", "label_1"]:
        zip_path = os.path.join(training_path, f"[라벨]KtelSpeech_train_{category}_{label}.zip")
        if not os.path.exists(zip_path):
            print(f"없음: {zip_path}")
            continue

        with zipfile.ZipFile(zip_path, 'r') as zf:
            for name in zf.namelist():
                if name.endswith('.txt'):
                    # 예: D60/J91/S00000001/0001.txt
                    parts = name.split('/')
                    if len(parts) != 4:
                        continue
                    d_cat, j_folder, session_id, filename = parts
                    unique_key = f"{d_cat}_{j_folder}_{session_id}"

                    with zf.open(name) as f:
                        text = f.read().decode('utf-8', errors='ignore')

                    sessions[unique_key][filename] = text

    return sessions

# D60 테스트
sessions_d60 = merge_category(training_path, "D60", "J91")
print("D60 병합된 세션 개수:", len(sessions_d60))

D60 병합된 세션 개수: 15192


In [25]:
import zipfile

def count_unique_sessions(zip_path):
    """zip 안에서 고유한 세션 폴더 개수 세기"""
    sessions = set()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for name in zf.namelist():
            parts = name.split('/')
            if len(parts) >= 3 and parts[2].startswith('S'):
                sessions.add(parts[2])
    return sessions

label0_zip = os.path.join(training_path, "[라벨]KtelSpeech_train_D60_label_0.zip")
label1_zip = os.path.join(training_path, "[라벨]KtelSpeech_train_D60_label_1.zip")

sessions_0 = count_unique_sessions(label0_zip)
sessions_1 = count_unique_sessions(label1_zip)

print("label_0 고유 세션 수:", len(sessions_0))
print("label_1 고유 세션 수:", len(sessions_1))
print("겹치는 세션 수:", len(sessions_0 & sessions_1))
print("합친 고유 세션 수:", len(sessions_0 | sessions_1))

label_0 고유 세션 수: 7596
label_1 고유 세션 수: 7596
겹치는 세션 수: 0
합친 고유 세션 수: 15192


In [26]:
import time

all_sessions = {}
all_sessions.update(sessions_d60)  # D60 이미 처리한 거 재사용

start = time.time()

sessions_d61 = merge_category(training_path, "D61", "J92")
sessions_d62 = merge_category(training_path, "D62", "J93")

all_sessions.update(sessions_d61)
all_sessions.update(sessions_d62)

elapsed = time.time() - start
print(f"D61+D62 처리 시간: {elapsed:.1f}초")
print(f"Training 전체 세션 수: {len(all_sessions)}")

D61+D62 처리 시간: 41.1초
Training 전체 세션 수: 49183


In [27]:
sessions_0_d61 = count_unique_sessions(os.path.join(training_path, "[라벨]KtelSpeech_train_D61_label_0.zip"))
sessions_1_d61 = count_unique_sessions(os.path.join(training_path, "[라벨]KtelSpeech_train_D61_label_1.zip"))
print("D61 합친 고유 세션 수:", len(sessions_0_d61 | sessions_1_d61))

sessions_0_d62 = count_unique_sessions(os.path.join(training_path, "[라벨]KtelSpeech_train_D62_label_0.zip"))
sessions_1_d62 = count_unique_sessions(os.path.join(training_path, "[라벨]KtelSpeech_train_D62_label_1.zip"))
print("D62 합친 고유 세션 수:", len(sessions_0_d62 | sessions_1_d62))

print("\nD60+D61+D62 원본 합계:", len(sessions_0 | sessions_1) + len(sessions_0_d61 | sessions_1_d61) + len(sessions_0_d62 | sessions_1_d62))
print("실제 병합된 Training 세션 수:", len(all_sessions))

D61 합친 고유 세션 수: 16084
D62 합친 고유 세션 수: 17907

D60+D61+D62 원본 합계: 49183
실제 병합된 Training 세션 수: 49183


In [28]:
def merge_category_validation(validation_path, category, subfolder):
    """Validation은 label_0만 있음"""
    sessions = defaultdict(dict)

    zip_path = os.path.join(validation_path, f"[라벨]KtelSpeech_valid_{category}_label_0.zip")
    if not os.path.exists(zip_path):
        print(f"없음: {zip_path}")
        return sessions

    with zipfile.ZipFile(zip_path, 'r') as zf:
        for name in zf.namelist():
            if name.endswith('.txt'):
                parts = name.split('/')
                if len(parts) != 4:
                    continue
                d_cat, j_folder, session_id, filename = parts
                unique_key = f"{d_cat}_{j_folder}_{session_id}_val"  # Training과 구분되게 _val 붙임

                with zf.open(name) as f:
                    text = f.read().decode('utf-8', errors='ignore')

                sessions[unique_key][filename] = text

    return sessions

start = time.time()

sessions_val_d60 = merge_category_validation(validation_path, "D60", "J91")
sessions_val_d61 = merge_category_validation(validation_path, "D61", "J92")
sessions_val_d62 = merge_category_validation(validation_path, "D62", "J93")

elapsed = time.time() - start
print(f"Validation 처리 시간: {elapsed:.1f}초")
print(f"D60 val: {len(sessions_val_d60)}, D61 val: {len(sessions_val_d61)}, D62 val: {len(sessions_val_d62)}")

Validation 처리 시간: 5.6초
D60 val: 189, D61 val: 231, D62 val: 236


In [29]:
# Training + Validation 전체 합치기
final_sessions = {}
final_sessions.update(all_sessions)  # Training
final_sessions.update(sessions_val_d60)
final_sessions.update(sessions_val_d61)
final_sessions.update(sessions_val_d62)

print("최종 전체 세션 수:", len(final_sessions))

# 각 세션의 조각들을 파일명 순서대로 정렬해서 하나로 합치기
def combine_session_text(file_dict):
    """{파일명: 텍스트} -> 파일명 순서대로 정렬해서 하나의 텍스트로 합침"""
    sorted_filenames = sorted(file_dict.keys())
    combined = " ".join(file_dict[fname].strip() for fname in sorted_filenames)
    return combined

# 저장할 경로
output_dir = "/content/상담음성_재병합"
os.makedirs(output_dir, exist_ok=True)

start = time.time()
for session_key, file_dict in final_sessions.items():
    combined_text = combine_session_text(file_dict)
    output_path = os.path.join(output_dir, f"{session_key}.txt")
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(combined_text)

elapsed = time.time() - start
print(f"저장 완료. 소요 시간: {elapsed:.1f}초")

# 최종 확인
saved_files = os.listdir(output_dir)
print("저장된 파일 개수:", len(saved_files))

최종 전체 세션 수: 49839
저장 완료. 소요 시간: 8.9초
저장된 파일 개수: 49839


In [30]:
import shutil

# 1. Drive에 폴더째로 복사 (압축 없이 원본 그대로 백업)
drive_backup_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성_재병합_원본"
shutil.copytree(output_dir, drive_backup_path)

print("Drive 백업 완료")
print("백업된 파일 개수:", len(os.listdir(drive_backup_path)))

KeyboardInterrupt: 

In [31]:
import shutil

# 중간에 멈췄으면 일부만 복사된 폴더는 지워서 깔끔하게 정리
drive_backup_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성_재병합_원본"
if os.path.exists(drive_backup_path):
    shutil.rmtree(drive_backup_path)
    print("일부 복사된 폴더 정리 완료")

일부 복사된 폴더 정리 완료


In [32]:
import os

output_dir = "/content/상담음성_재병합"
print("존재?", os.path.exists(output_dir))
print("파일 개수:", len(os.listdir(output_dir)))

존재? True
파일 개수: 49839


In [33]:
import zipfile
import time

zip_output_path = "/content/drive/MyDrive/voice_pishing_text_merged/상담음성_재병합_결과.zip"

start = time.time()
with zipfile.ZipFile(zip_output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(output_dir):
        file_path = os.path.join(output_dir, fname)
        zf.write(file_path, arcname=fname)
elapsed = time.time() - start

print(f"압축 완료, {elapsed:.1f}초 소요")

with zipfile.ZipFile(zip_output_path, 'r') as zf:
    names_in_zip = [n for n in zf.namelist() if n.endswith('.txt')]

print("zip 안 실제 txt 파일 개수:", len(names_in_zip))

압축 완료, 49.1초 소요
zip 안 실제 txt 파일 개수: 49839


In [34]:
finance_keywords = ['은행', '계좌', '대출', '카드', '이체', '통장', '적금', '예금', '금융', '보험']

def is_finance_related(text):
    return any(kw in text for kw in finance_keywords)

import time
start = time.time()

finance_files = []
for fname in os.listdir(output_dir):
    if not fname.endswith('.txt'):
        continue
    path = os.path.join(output_dir, fname)
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    if is_finance_related(text):
        finance_files.append(fname)

elapsed = time.time() - start
print(f"필터링 완료, {elapsed:.1f}초 소요")
print("금융 관련 파일 개수:", len(finance_files))

필터링 완료, 6.4초 소요
금융 관련 파일 개수: 29087


In [35]:
import random

# 금융 관련 정상 데이터에서 239개 랜덤 샘플링 (사기 데이터 개수와 동일하게)
random.seed(42)
finance_sample_files = random.sample(finance_files, 239)

normal_finance_data = []
for fname in finance_sample_files:
    path = os.path.join(output_dir, fname)
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    normal_finance_data.append({'text': text, 'label': 0, 'source': fname})

print("금융 정상 데이터:", len(normal_finance_data))
print("사기 데이터:", len(fraud_data))

# 5:5 데이터프레임 구성
all_data_finance = fraud_data + normal_finance_data
df_finance = pd.DataFrame(all_data_finance)
print(df_finance['label'].value_counts())

# 정규화 적용 (라벨까지 완전 제거하는 버전으로)
df_finance['text_normalized'] = df_finance['text'].apply(normalize_finance_text)
df_finance['text_no_label'] = df_finance['text_normalized'].apply(strip_speaker_label)

# Train/Val 분할
train_df5, val_df5 = train_test_split(
    df_finance, test_size=0.2, random_state=42, stratify=df_finance['label']
)

print("Train:", len(train_df5), "Val:", len(val_df5))

금융 정상 데이터: 239
사기 데이터: 239
label
1    239
0    239
Name: count, dtype: int64
Train: 382 Val: 96


In [36]:
train_encodings5 = tokenizer(
    train_df5['text_no_label'].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)
val_encodings5 = tokenizer(
    val_df5['text_no_label'].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)

train_dataset5 = VoicePhishingDataset(train_encodings5, train_df5['label'].tolist())
val_dataset5 = VoicePhishingDataset(val_encodings5, val_df5['label'].tolist())

model5 = ElectraForSequenceClassification.from_pretrained(
    "skplanet/dialog-koelectra-small-discriminator",
    num_labels=2
)

trainer5 = Trainer(
    model=model5,
    args=training_args,
    train_dataset=train_dataset5,
    eval_dataset=val_dataset5,
    compute_metrics=compute_metrics,
)

trainer5.train()

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at skplanet/dialog-koelectra-small-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.649100,0.540310,0.979167,1.000000,0.958333,0.978723
2,0.448100,0.344642,0.989583,1.000000,0.979167,0.989474
3,0.308900,0.286639,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=72, training_loss=0.47382266488340163, metrics={'train_runtime': 8.4763, 'train_samples_per_second': 135.2, 'train_steps_per_second': 8.494, 'total_flos': 8428727098368.0, 'train_loss': 0.47382266488340163, 'epoch': 3.0})

In [37]:
import torch.nn.functional as F

model5.eval()
predictions_info5 = []

with torch.no_grad():
    for i in range(len(val_dataset5)):
        item = val_dataset5[i]
        input_ids = item['input_ids'].unsqueeze(0).to(model5.device)
        attention_mask = item['attention_mask'].unsqueeze(0).to(model5.device)
        label = item['labels'].item()

        output = model5(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(output.logits, dim=1)
        confidence = probs.max().item()

        predictions_info5.append({'true_label': label, 'confidence': confidence})

result_df5 = pd.DataFrame(predictions_info5)
print(result_df5['confidence'].describe())

count    96.000000
mean      0.751319
std       0.026973
min       0.568294
25%       0.747568
50%       0.749853
75%       0.766958
max       0.772909
Name: confidence, dtype: float64


In [41]:
!git config --global user.name "jinyeong-731"
!git config --global user.email "pjinhyeong731@gmail.com"

In [39]:
!git remote set-url origin https://github.com/aihuman-7th/proj1-e.git

fatal: not a git repository (or any of the parent directories): .git


In [42]:
!git add .
!git commit -m "sk플래닛 우리데이터 학습 및 작업 코드 업로드"

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
